# 02 · Example: using `src/` in a model notebook

This is a template every model notebook (scVI, VAE, baseline...) can copy.
It does the whole evaluation plumbing by importing the shared `src/` modules instead of
re-writing the splits, so nobody reimplements folds or leaks the test set.

The only thing you change is the box marked `### YOUR MODEL HERE`. Everything around it
(loading data, attaching the official folds, HVG-on-train-only, metrics) stays identical across
the team, so our numbers are comparable.

First run `01_splits_and_eval_protocol.ipynb` once first, it generates
`data/splits.parquet` that this notebook loads.

### 0. Imports
The `src/` package holds the reusable logic (see `splits.py`, `data.py`, `hvg.py`, `metrics.py`).

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd()))   

from src import data as D       
from src import hvg as H        
from src import metrics as M    
from src import splits as S     
import numpy as np, pandas as pd
print("src loaded · SEED =", S.SEED)

/private/tmp/claude-501/-Users-almudenamartin/5d33a055-c5d7-4596-b2bd-05c9a1d29552/scratchpad/venv/lib/python3.12/site-packages/numcodecs/__init__.py:108: DeprecationWarning: crc32c usage is deprecated since numcodecs v0.16.4. It is recommended to install google_crc32c instead.
  from numcodecs.checksum32 import CRC32, Adler32, JenkinsLookup3


src loaded · SEED = 42


### 1. Load data and attach the folds

`load_adata()` finds `wu2021_brca.h5ad` automatically and fixes the layout: in the file `.X` is
already log1p-CP10k and the counts are in `.raw`, so after loading `adata.X` = **raw integer counts**
(what scVI/MrVI and seurat_v3 HVG need) and `adata.layers["lognorm"]` = log1p-CP10k (for PCA-like
baselines). Never apply `log1p` again. `attach_folds()` reads
`data/splits.parquet` (the 26-patient table from notebook 01) and maps each patient's
`fold_5 / fold_lopo / fold_batch` onto every cell via `donor_id`.

> The parquet is not the data, it only carries the patient->fold map. The cells and genes
> come from the h5ad. This cell is where the two are joined.

In [2]:
adata = D.load_adata()  # the real data: cells x genes
D.attach_folds(adata) # adds obs['fold_5'], obs['fold_lopo'], obs['fold_batch']
print(adata)
adata.obs[['donor_id','fold_5','fold_lopo','fold_batch']].drop_duplicates('donor_id').head()

AnnData object with n_obs × n_vars = 100064 × 28523
    obs: 'donor_id', 'percent_mito', 'nCount_RNA', 'nFeature_RNA', 'celltype_major', 'celltype_minor', 'celltype_subset', 'subtype', 'gene_module', 'calls', 'normal_cell_call', 'CNA_value', 'batch_run', 'multiplexed', 'cryo_state', 'development_stage_ontology_term_id', 'cancer_type', 'ER', 'PR', 'HER2_IHC', 'HER2_ISH', 'HER2_ISH_ratio', 'Ki67', 'subtype_by_IHC', 'treatment_status', 'treatment_details', 'assay_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'disease_ontology_term_id', 'grade', 'cell_type_ontology_term_id', 'tissue_type', 'HER2_ISH_legacy', 'cell_type_ontology_term_id_legacy', 'celltype_major_legacy', 'celltype_minor_legacy', 'celltype_subset_legacy', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'fold_5', 'fold_lopo', 'fold_batch'
    var: 'fea

,donor_id,fold_5,fold_lopo,fold_batch
CID3586_AAGACCTCAGCATGAG,CID3586,3,0,13
CID3921_AAAGATGGTGAGGCTA,CID3921,2,2,12
CID45171_CATCAAGTCAACACCA,CID45171,4,25,9
CID3838_AAACGCTAGAGCCGAT,CID3838,1,1,16
CID4066_ACTGCTCTCTCGATGA,CID4066,0,8,12


### 2. Fold

`fold_mask(adata, scheme, test_value)` returns boolean `train_mask, test_mask` with no cell
of a patient split across the two. We then select HVG on train only (`select_hvg_train`)
so feature selection never sees the test patient.

In [3]:
SCHEME, TEST_FOLD = "fold_5", 0
train_mask, test_mask = D.fold_mask(adata, SCHEME, TEST_FOLD)
print(f"{SCHEME}=={TEST_FOLD}:  train cells = {train_mask.sum():,}   test cells = {test_mask.sum():,}")

hvg = H.select_hvg_train(adata, train_mask, n_top=2000) # computed on TRAIN cells only
print("HVG selected on train:", len(hvg))

fold_5==0:  train cells = 78,344   test cells = 21,720


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


HVG selected on train: 2000


### 3. `### YOUR MODEL HERE`

Fit your model on the train cells / HVG only, then produce a latent embedding
(count models such as scVI use `adata.X`; Gaussian ones such as PCA use `layers["lognorm"]`)
`Z` for all cells. 
Below is a placeholder (PCA) just so the template runs end-to-end, but it can be replaced with other.

In [4]:
# ---- placeholder embedding ----
from sklearn.decomposition import PCA
X = adata[:, hvg].layers["lognorm"]   # already log1p-CP10k: no extra log
X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
Z_train = PCA(n_components=30, random_state=S.SEED).fit(X[train_mask])
Z = Z_train.transform(X)  # embedding for ALL cells (train fit, applied to all)
print("Z:", Z.shape)
# ------------------------------------------------------------------------

Z: (100064, 30)


### 4. Evaluate on the test fold

All metrics live in `src/metrics.py`, so all of us compute them the same way:
- `clustering_scores` -> ARI / NMI / cell-type silhouette (biological signal kept),
- `batch_entropy` -> patient mixing (batch correction worked).

In [5]:
te = test_mask
clu = M.clustering_scores(Z[te], adata.obs['celltype_major'].values[te])
mix = M.batch_entropy(Z[te], adata.obs['donor_id'].values[te], k=30)
print("test fold", TEST_FOLD, "->", {k: round(v,3) for k,v in clu.items()}, "| patient_mixing:", round(mix,3))

test fold 0 -> {'ARI': 0.752, 'NMI': 0.807, 'ASW_celltype': 0.643} | patient_mixing: 0.225


### 5. Loop over every fold (cross-validation)

`iter_folds` yields `(test_value, train_mask, test_mask)` for the whole scheme. 

In [6]:
rows = []
for test_value, tr, te in D.iter_folds(adata, scheme="fold_5"):
    hvg_k = H.select_hvg_train(adata, tr, n_top=2000)
    Xk = adata[:, hvg_k].layers["lognorm"]
    Xk = Xk.toarray() if hasattr(Xk,"toarray") else np.asarray(Xk)
    Zk = PCA(n_components=30, random_state=S.SEED).fit(Xk[tr]).transform(Xk) # <-- your model here too
    s  = M.clustering_scores(Zk[te], adata.obs['celltype_major'].values[te])
    s['patient_mixing'] = M.batch_entropy(Zk[te], adata.obs['donor_id'].values[te])
    s['fold'] = test_value
    rows.append(s)

res = pd.DataFrame(rows).set_index('fold')
print(res.round(3))
print("\nMEAN:", res.mean(numeric_only=True).round(3).to_dict())

        ARI    NMI  ASW_celltype  patient_mixing
fold                                            
3     0.538  0.696         0.625           0.252
2     0.544  0.709         0.638           0.138
4     0.677  0.786         0.666           0.158
1     0.632  0.743         0.657           0.188
0     0.752  0.807         0.643           0.225

MEAN: {'ARI': 0.629, 'NMI': 0.748, 'ASW_celltype': 0.646, 'patient_mixing': 0.192}


### Conclusion
Copy this notebook, replace the two `### YOUR MODEL HERE` boxes with your model,
and we get comparable and leak free CV metrics automatically because the splits, HVG rule and
metrics all come from the shared `src/`.